In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 07 Quality Performance Index
direction adjustment -> percentile rank -> weighted mean over available measures (>= 8 of 11) -> tercile categories.

In [2]:
m = pd.read_csv(PROC/'irf_facilities_merged.csv', dtype={CCN:str}, parse_dates=['Certification Date'])
df, Pm, sens, wtab, contrib, val, info = P.stage_index(m); info

{'n_scored': 1123,
 'n_excluded_lt_min': 99,
 'min_measures': 8,
 'tercile_cuts': [45.296750680512474, 55.14681671098239],
 'class_counts': {'Low': 375, 'Medium': 374, 'High': 374},
 'index_mean': 49.938304131164955,
 'index_sd': 10.88592091262153,
 'index_min': 17.560887355344256,
 'index_max': 82.9144468880136,
 'measures_available_dist': {0: 77,
  1: 7,
  3: 2,
  4: 2,
  5: 2,
  6: 7,
  7: 2,
  8: 4,
  9: 60,
  10: 163,
  11: 896}}

In [3]:
wtab

,measure,better,domain,Equal weights,Domain-balanced,PCA (PC1) loadings
I_015_01_SIR,C. difficile infection SIR,lower,Safety,1.0,0.0667,0.1548
I_013_01_OBS_RATE,Falls with major injury (%),lower,Safety,1.0,0.0667,0.0441
I_022_01_ADJ_RATE,"New/worsened pressure ulcers, risk-adj. (%)",lower,Safety,1.0,0.0667,0.0815
I_017_01_PPR_PD_RSRR,Preventable 30-day post-discharge readmission ...,lower,Readmission & community return,1.0,0.0667,0.2446
I_018_01_PPR_WI_RSRR,Preventable within-stay readmission (RSRR),lower,Readmission & community return,1.0,0.0667,0.3594
I_019_02_DTC_RS_RATE,"Discharge to community, risk-std. (%)",higher,Readmission & community return,1.0,0.0667,0.4187
I_026_01_OBS_RATE,Discharge function score met/exceeded (%),higher,Functional outcome,1.0,0.2000,0.1755
I_021_01_OBS_RATE,Drug regimen review with follow-up (%),higher,Care processes,1.0,0.0667,0.3277
I_024_01_OBS_RATE,Med list to next care setting (%),higher,Care processes,1.0,0.0667,0.4138
I_025_02_OBS_RATE,Med list to patient/caregiver (%),higher,Care processes,1.0,0.0667,0.4479


In [4]:
contrib  # how strongly each measure's percentile tracks the final index

,measure,n_available,spearman_with_index
0,C. difficile infection SIR,977,0.134100
1,Falls with major injury (%),1135,0.251104
2,"New/worsened pressure ulcers, risk-adj. (%)",1135,0.292775
3,Preventable 30-day post-discharge readmission ...,1126,0.367473
4,Preventable within-stay readmission (RSRR),1126,0.468817
5,"Discharge to community, risk-std. (%)",1126,0.557962
6,Discharge function score met/exceeded (%),1134,0.281992
7,Drug regimen review with follow-up (%),1135,0.425443
8,Med list to next care setting (%),1126,0.460593
9,Med list to patient/caregiver (%),1027,0.525376


### Sensitivity to the weighting scheme

In [5]:
sens

,alternative,spearman_vs_equal,category_agreement_%,cohen_kappa
0,Domain-balanced,0.904662,73.642030,0.604630
1,PCA (PC1) loadings,0.941780,79.964381,0.699465


### Descriptive comparison with CMS's own per-measure label (discharge to community). Partly circular (that measure is inside the index) -> descriptive only.

In [6]:
val